# Advanced 08 — Brownian Motion and Introduction to Stochastic Calculus

Practice notebook for the **"Brownian Motion and Introduction to Stochastic Calculus"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Standard Brownian motion

A process \((B_t)_{t\geq 0}\) is a **standard Brownian motion** if:

1. \(B_0 = 0\) almost surely.
2. \(B_t\) has **independent increments**.
3. For \(0 \leq s < t\), \(B_t - B_s \sim N(0,\, t-s)\).
4. \(B_t\) has **continuous sample paths** almost surely.

Because increments over disjoint intervals are independent Gaussians, we can
simulate a path on a grid \(0 = t_0 < t_1 < \dots < t_n = T\) by accumulating
independent increments:

$$
B_{t_{k+1}} = B_{t_k} + \sqrt{t_{k+1} - t_k}\, Z_k, \qquad Z_k \sim N(0,1)\ \text{i.i.d.}
$$

Below we simulate many paths and verify the marginal moments

$$
E[B_t] = 0, \qquad \mathrm{Var}(B_t) = t, \qquad \mathrm{Cov}(B_s, B_t) = \min(s, t).
$$

**Your turn:** change `T` and `N_PATHS` and confirm the empirical moments still match.


In [ ]:
T = 1.0      # horizon
N = 500      # grid points per path
N_PATHS = 4000
dt = T / N
t_grid = np.linspace(0.0, T, N + 1)

# Independent Gaussian increments ~ N(0, dt); cumulative sum gives B_t.
rng = np.random.default_rng(42)
dW = rng.normal(loc=0.0, scale=np.sqrt(dt), size=(N_PATHS, N))
B = np.concatenate([np.zeros((N_PATHS, 1)), np.cumsum(dW, axis=1)], axis=1)  # (N_PATHS, N+1)

# Pick a few times to check marginals.
check_t = [0.25, 0.5, 0.75, 1.0]
print(f"{'t':>6} {'E[B_t]':>10} {'Var(B_t)':>10} {'theory Var':>12}")
for tt in check_t:
    k = int(round(tt / dt))
    col = B[:, k]
    print(f"{tt:6.2f} {col.mean():10.4f} {col.var(ddof=1):10.4f} {tt:12.4f}")

# Cov(B_s, B_t) = min(s, t): empirical vs theory for a grid of (s, t) pairs.
s_vals = np.array([0.2, 0.4, 0.6, 0.8])
t_vals = np.array([0.3, 0.5, 0.7, 1.0])
print("\nCov(B_s, B_t) empirical vs min(s, t):")
print(f"{'s':>5} {'t':>5} {'emp cov':>10} {'min(s,t)':>10}")
for s in s_vals:
    ks = int(round(s / dt))
    for t in t_vals:
        kt = int(round(t / dt))
        emp = np.cov(B[:, ks], B[:, kt])[0, 1]
        print(f"{s:5.2f} {t:5.2f} {emp:10.4f} {min(s, t):10.4f}")

# Plot a few sample paths.
fig, ax = plt.subplots()
for p in range(8):
    ax.plot(t_grid, B[p], lw=0.8, alpha=0.8)
ax.axhline(0.0, color="k", lw=0.5)
ax.set_xlabel("t"); ax.set_ylabel("B_t"); ax.set_title("Sample Brownian paths")
plt.show()


---
## Part 2 — Quadratic variation

For a partition \(\Pi_n = \{0 = t_0 < t_1 < \dots < t_n = t\}\), the realized
**quadratic variation** along a path is

$$
V_t(\Pi_n) = \sum_{k=0}^{n-1} \bigl(B_{t_{k+1}} - B_{t_k}\bigr)^2.
$$

As the mesh \(\|\Pi_n\| \to 0\), \(V_t(\Pi_n) \to t\). We write
\([B]_t = t\). This is the *roughness* of Brownian motion: a smooth
differentiable function has **zero** quadratic variation, because its increments
shrink faster than their squares accumulate.

Below we (a) compute \(V_T(\Pi_n)\) on a fine grid and watch it concentrate
near \(T\) as the mesh refines, and (b) contrast with the quadratic variation of
a smooth function \(f(t) = \sin(\pi t)\), which collapses to 0.

**Your turn:** add a coarser partition (e.g. `N = 20`) and see \(V_T\) become
noisier — convergence in probability, not pathwise determinism.


In [ ]:
T = 1.0
mesh_list = [50, 200, 1000, 5000]
N_PATHS = 3000

rng = np.random.default_rng(7)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# (a) Brownian QV vs mesh refinement.
means, stds = [], []
for N in mesh_list:
    dt = T / N
    dW = rng.normal(0.0, np.sqrt(dt), size=(N_PATHS, N))
    qv = (dW ** 2).sum(axis=1)            # sum of squared increments = V_T(Pi_n)
    means.append(qv.mean()); stds.append(qv.std(ddof=1))
    print(f"N={N:5d}  mean V_T={qv.mean():.4f}  std V_T={qv.std(ddof=1):.4f}  (theory: T={T:.4f})")

axes[0].errorbar(mesh_list, means, yerr=stds, fmt="o-", capsize=3, label="empirical V_T")
axes[0].axhline(T, color="r", ls="--", label="theoretical [B]_T = T")
axes[0].set_xscale("log"); axes[0].set_xlabel("mesh N (log)"); axes[0].set_ylabel("V_T")
axes[0].set_title("Brownian QV converges to T"); axes[0].legend()

# (b) Smooth function has zero QV as mesh refines.
t_fine = np.linspace(0.0, T, 10000)
f = np.sin(np.pi * t_fine)
qv_smooth = []
for N in mesh_list:
    grid = np.linspace(0.0, T, N + 1)
    fg = np.sin(np.pi * grid)
    qv_smooth.append(np.sum(np.diff(fg) ** 2))
    print(f"smooth  N={N:5d}  QV={qv_smooth[-1]:.6e}")
axes[1].plot(mesh_list, qv_smooth, "s-", label="QV of sin(pi t)")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("mesh N (log)"); axes[1].set_ylabel("sum of squared increments")
axes[1].set_title("Smooth function QV -> 0"); axes[1].legend()
plt.tight_layout(); plt.show()


---
## Part 3 — It\^o integral via left-endpoint Riemann sums

For an adapted process \((H_t)\), the **It\^o integral** is the \(L^2\)-limit
of stochastic Riemann sums that evaluate the integrand at the **left endpoint** of
each subinterval:

$$
\int_0^t H_s\, dB_s \;=\; L^2\!\lim_{\|\Pi_n\|\to 0} \sum_{k} H_{t_k}\,\bigl(B_{t_{k+1}} - B_{t_k}\bigr).
$$

Two consequences we verify empirically:

* **Martingale / zero mean:** \(E\!\left[\int_0^t H_s\,dB_s\right] = 0\) when
  \(H\) is adapted and integrable.
* **It\^o isometry:**
  $$E\!\left[\left(\int_0^t H_s\,dB_s\right)^{\!2}\right] = E\!\left[\int_0^t H_s^2\,ds\right].$$

We use the simple adapted integrand \(H_s = B_s\) on \([0, T]\), for which the
closed form is \(\int_0^T B_s\,dB_s = \tfrac12 B_T^2 - \tfrac12 T\). The
left-endpoint rule should reproduce this (up to discretization error) and satisfy
both properties above.

**Your turn:** swap in \(H_s = s\) (a deterministic integrand). Predict
\(E[\int_0^T s\,dB_s]\) and its variance before running.


In [ ]:
T = 1.0
N = 1000
N_PATHS = 5000
dt = T / N
t_grid = np.linspace(0.0, T, N + 1)

rng = np.random.default_rng(123)
dW = rng.normal(0.0, np.sqrt(dt), size=(N_PATHS, N))
B = np.concatenate([np.zeros((N_PATHS, 1)), np.cumsum(dW, axis=1)], axis=1)

# It^o sum: left-endpoint H_{t_k} * (B_{t_{k+1}} - B_{t_k}), with H_s = B_s.
H_left = B[:, :-1]                       # H evaluated at left endpoint of each cell
ito_sum = (H_left * dW).sum(axis=1)      # pathwise Ito integral approximation

# Closed form for int_0^T B_s dB_s = 0.5*B_T^2 - 0.5*T
closed_form = 0.5 * B[:, -1]**2 - 0.5 * T

# Continuous isometry RHS: int_0^T B_s^2 ds (trapezoidal-ish Riemann sum of B^2)
iso_rhs = ((B[:, :-1] ** 2) * dt).sum(axis=1)

print("E[Ito integral]            :", ito_sum.mean(), "  (theory: 0)")
print("Var(Ito integral)          :", ito_sum.var(ddof=1), "  (theory: E[int B^2 ds])")
print("E[int B_s^2 ds]            :", iso_rhs.mean())
print("|Var - E[int B^2 ds]|      :", abs(ito_sum.var(ddof=1) - iso_rhs.mean()))
print()
print("mean |ito_sum - closed|    :", np.mean(np.abs(ito_sum - closed_form)))
print("max  |ito_sum - closed|    :", np.max(np.abs(ito_sum - closed_form)))

fig, ax = plt.subplots()
ax.hist(ito_sum, bins=60, density=True, alpha=0.6, label="Ito sum")
ax.hist(closed_form, bins=60, density=True, alpha=0.4, label="0.5 B_T^2 - 0.5 T")
ax.set_xlabel("value"); ax.set_ylabel("density")
ax.set_title("Ito integral vs closed form for H_s = B_s"); ax.legend()
plt.show()


---
## Part 4 — It\^o's formula

For an It\^o process \(dX_t = \mu\,dt + \sigma\,dB_t\) and a
\(C^{1,2}\) function \(f(t, x)\),

$$
df(t, X_t) = \left( f_t + \mu f_x + \tfrac{1}{2}\sigma^2 f_{xx} \right) dt
           + \sigma f_x\, dB_t.
$$

The extra term \(\tfrac12 \sigma^2 f_{xx}\, dt\) is the signature of
stochastic calculus — it comes from the non-zero quadratic variation of \(X\).

We verify it on the canonical example \(f(x) = x^2\) with \(X_t = B_t\$
(notation: we write \(W\) in code, same object). The formula gives

$$
d(B_t^2) = 2 B_t\, dB_t + dt,
\quad\text{i.e.}\quad
B_T^2 - B_0^2 = \int_0^T 2 B_s\, dB_s + \int_0^T 1\, ds.
$$

Rearranging, \(\int_0^T 2 B_s\, dB_s = B_T^2 - T\). We compute the left-hand
Ito sum directly and compare it to the right-hand side \(B_T^2 - T\). The
equality should hold up to \(O(\text{mesh})\) discretization error that vanishes
as the grid refines.

**Your turn:** repeat for \(f(x) = x^3\), where Ito's formula gives
\(d(B_t^3) = 3 B_t^2\, dB_t + 3 B_t\, dt\).


In [ ]:
T = 1.0
N = 2000
N_PATHS = 5000
dt = T / N

rng = np.random.default_rng(2024)
dW = rng.normal(0.0, np.sqrt(dt), size=(N_PATHS, N))
B = np.concatenate([np.zeros((N_PATHS, 1)), np.cumsum(dW, axis=1)], axis=1)

# Ito expansion of d(B^2) = 2 B dB + dt, integrated over [0, T]:
#   int_0^T 2 B_s dB_s = B_T^2 - T.
ito_lhs = (2.0 * B[:, :-1] * dW).sum(axis=1)   # left-endpoint Ito sum
ito_rhs = B[:, -1]**2 - T                      # B_T^2 - T

print("mean(Ito LHS)            :", ito_lhs.mean(), "  (theory: E[B_T^2 - T] = 0)")
print("mean(B_T^2 - T)          :", ito_rhs.mean())
print("mean |LHS - RHS|         :", np.mean(np.abs(ito_lhs - ito_rhs)))
print("std  |LHS - RHS|         :", np.std(np.abs(ito_lhs - ito_rhs), ddof=1))

# Refinement study: discretization error should shrink like O(sqrt(dt)) pathwise.
print("\nRefinement study (mean abs error vs mesh):")
for N_test in [100, 400, 1600, 6400, 25600]:
    dt_t = T / N_test
    dWt = rng.normal(0.0, np.sqrt(dt_t), size=(N_PATHS, N_test))
    Bt = np.concatenate([np.zeros((N_PATHS, 1)), np.cumsum(dWt, axis=1)], axis=1)
    lhs = (2.0 * Bt[:, :-1] * dWt).sum(axis=1)
    rhs = Bt[:, -1]**2 - T
    print(f"  N={N_test:6d}  dt={dt_t:.6f}  mean|err|={np.mean(np.abs(lhs - rhs)):.6e}")

fig, ax = plt.subplots()
ax.scatter(ito_rhs, ito_lhs, s=4, alpha=0.3)
lims = [ito_rhs.min(), ito_rhs.max()]
ax.plot(lims, lims, "r--", lw=1, label="y = x (Ito formula)")
ax.set_xlabel("B_T^2 - T  (RHS)"); ax.set_ylabel("Ito sum 2 B dB  (LHS)")
ax.set_title("Ito's formula for f(B) = B^2"); ax.legend()
plt.show()


---
## Part 5 — Geometric Brownian motion (application of Ito's formula)

Applying Ito's formula to \(f(x) = \log x\) with \(dS_t = \mu S_t\,dt + \sigma S_t\,dB_t\)
yields

$$
d \log S_t = \left(\mu - \tfrac{1}{2}\sigma^2\right) dt + \sigma\, dB_t,
\quad\text{so}\quad
S_t = S_0 \exp\!\left( \left(\mu - \tfrac12 \sigma^2\right) t + \sigma B_t \right).
$$

This is geometric Brownian motion (GBM), the Black--Scholes price model: \(S_t\)
is **log-normal**. We simulate the SDE directly with the Euler--Maruyama scheme
(left-endpoint discretization — the same idea as the Ito sum) and compare to the
exact closed-form solution.

**Your turn:** which parameter controls volatility skew — \(\mu\) or
\(\sigma\)? Plot the log-density of \(\log S_T\) and overlay the exact
\(N\!\left((\mu - \tfrac12\sigma^2)T,\ \sigma^2 T\right)\) density.


In [ ]:
mu = 0.08
sigma = 0.30
S0 = 100.0
T = 1.0
N = 1000
N_PATHS = 5000
dt = T / N

rng = np.random.default_rng(99)
dW = rng.normal(0.0, np.sqrt(dt), size=(N_PATHS, N))

# Euler-Maruyama: S_{k+1} = S_k + mu*S_k*dt + sigma*S_k*dW_k  (Ito left-endpoint).
S = np.zeros((N_PATHS, N + 1))
S[:, 0] = S0
for k in range(N):
    S[:, k + 1] = S[:, 0] if False else S[:, k] + mu * S[:, k] * dt + sigma * S[:, k] * dW[:, k]

# Exact GBM solution: S_T = S0 * exp((mu - 0.5 sigma^2) T + sigma B_T).
B_T = dW.sum(axis=1)
S_T_exact = S0 * np.exp((mu - 0.5 * sigma**2) * T + sigma * B_T)

print("Euler-Maruyama  mean S_T :", S[:, -1].mean())
print("Exact GBM       mean S_T :", S_T_exact.mean(), " (theory:", S0 * np.exp(mu * T), ")")
print("median abs rel error    :", np.median(np.abs(S[:, -1] - S_T_exact) / S_T_exact))

log_ST = np.log(S[:, -1])
mean_theory = (mu - 0.5 * sigma**2) * T
std_theory = sigma * np.sqrt(T)
print(f"\nlog S_T: empirical mean={log_ST.mean():.4f} (theory {mean_theory:.4f}), "
      f"std={log_ST.std(ddof=1):.4f} (theory {std_theory:.4f})")

fig, ax = plt.subplots()
ax.hist(log_ST, bins=60, density=True, alpha=0.6, label="simulated log S_T")
xs = np.linspace(log_ST.min(), log_ST.max(), 400)
ax.plot(xs, stats.norm.pdf(xs, loc=mean_theory, scale=std_theory),
        "r-", lw=2, label="exact N((mu-sig^2/2)T, sig^2 T)")
ax.set_xlabel("log S_T"); ax.set_ylabel("density")
ax.set_title("GBM log-returns are Gaussian"); ax.legend()
plt.show()


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Simulate \(M = 10{,}000\) Brownian paths on \([0, 2]\) with \(N = 800\)
steps. Verify empirically that
\(E[B_t] = 0\), \(\mathrm{Var}(B_t) = t\), and
\(\mathrm{Cov}(B_s, B_t) = \min(s, t)\) for at least four pairs \((s, t)\).
Report the maximum absolute error across all checks.

2. Compute the realized quadratic variation \(V_T(\Pi_n)\) of Brownian
motion on \([0, 1]\) for \(n \in \{100, 1000, 10{,}000\}\) over \(3000\)
paths. Show that the *standard deviation* of \(V_T\) across paths shrinks like
\(\sqrt{2T/n}\) (theory: \(\sum (\Delta B)^2\) is a sum of i.i.d. scaled
\(\chi^2_1\) variables with variance \(2\,\Delta t\) per term). Contrast with
the QV of \(f(t) = t^2\) on the same grids.

3. For the adapted integrand \(H_s = B_s^2\) on \([0, 1]\), approximate
\(\int_0^1 B_s^2\, dB_s\) with a left-endpoint Ito sum over \(5000\) paths and
\(N = 2000\) steps. Verify (i) \(E[\int B_s^2\, dB_s] = 0\), and (ii) the Ito
isometry \(\mathrm{Var}(\int B_s^2\, dB_s) = E[\int_0^1 B_s^4\, ds]\).
Estimate the RHS by a separate Riemann sum and compare.

4. Apply Ito's formula to \(f(x) = x^4\) with \(X_t = B_t\). Derive
\(d(B_t^4) = 4 B_t^3\, dB_t + 6 B_t^2\, dt\), so
\(\int_0^T 4 B_s^3\, dB_s = B_T^4 - 2 T^2 - 6 \int_0^T B_s^2\, ds\)
(use \(E[B_t^2] = t\) and \(E[B_t^4] = 3 t^2\) to identify \(E[B_T^4] = 3T^2\)).
Confirm the identity numerically over \(5000\) paths with \(N = 2000\).

5. Simulate geometric Brownian motion with \(\mu = 0.10\),
\(\sigma = 0.40\), \(S_0 = 50\), \(T = 1\), using both Euler--Maruyama
(left-endpoint, Ito) and the exact closed form
\(S_T = S_0 \exp((\mu - \tfrac12 \sigma^2) T + \sigma B_T)\) over \(10{,}000\)
paths. Compare the empirical mean, variance, and the 5%/95% quantiles of
\(S_T\) between the two schemes. Which moments match better, and why does
Euler--Maruyama introduce a small bias in \(E[S_T]\)?


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Simulate and check moments:

```python
T, N, M = 2.0, 800, 10000
dt = T / N
rng = np.random.default_rng(1)
dW = rng.normal(0.0, np.sqrt(dt), size=(M, N))
B = np.concatenate([np.zeros((M, 1)), np.cumsum(dW, axis=1)], axis=1)
t_grid = np.linspace(0.0, T, N + 1)

max_err = 0.0
for tt in [0.5, 1.0, 1.5, 2.0]:
    k = int(round(tt / dt))
    col = B[:, k]
    max_err = max(max_err, abs(col.mean() - 0.0))
    max_err = max(max_err, abs(col.var(ddof=1) - tt))

pairs = [(0.3, 0.7), (0.5, 1.5), (0.8, 1.2), (1.0, 2.0)]
for s, t in pairs:
    ks, kt = int(round(s / dt)), int(round(t / dt))
    emp = np.cov(B[:, ks], B[:, kt])[0, 1]
    max_err = max(max_err, abs(emp - min(s, t)))
print("max abs error:", max_err)
```
Expected `max abs error` of order \(1/\sqrt{M} \approx 0.01\).

**2.** QV shrinking variance and smooth contrast:

```python
T, N_PATHS = 1.0, 3000
rng = np.random.default_rng(2)
for n in [100, 1000, 10000]:
    dt = T / n
    dW = rng.normal(0.0, np.sqrt(dt), size=(N_PATHS, n))
    qv = (dW ** 2).sum(axis=1)
    theory_sd = np.sqrt(2 * T / n)
    print(f"n={n:6d}  mean={qv.mean():.4f}  std={qv.std(ddof=1):.4f}  theory std={theory_sd:.4f}")

# Smooth f(t)=t^2 has zero QV in the limit.
for n in [100, 1000, 10000]:
    g = np.linspace(0, T, n + 1) ** 2
    print(f"smooth n={n:6d}  QV={np.sum(np.diff(g)**2):.3e}")
```
Brownian QV std tracks \(\sqrt{2T/n}\); smooth QV vanishes polynomially.

**3.** Ito integral of \(B_s^2\) with isometry:

```python
T, N, M = 1.0, 2000, 5000
dt = T / N
rng = np.random.default_rng(3)
dW = rng.normal(0.0, np.sqrt(dt), size=(M, N))
B = np.concatenate([np.zeros((M, 1)), np.cumsum(dW, axis=1)], axis=1)

ito = (B[:, :-1]**2 * dW).sum(axis=1)            # int B^2 dB (left endpoint)
rhs = ((B[:, :-1]**4) * dt).sum(axis=1)          # int B^4 ds (Riemann)
print("E[Ito]            :", ito.mean(), "  (theory 0)")
print("Var(Ito)          :", ito.var(ddof=1))
print("E[int B^4 ds]     :", rhs.mean(), "  (isometry RHS)")
print("|Var - E[int B^4]|:", abs(ito.var(ddof=1) - rhs.mean()))
```

**4.** Ito's formula for \(B^4\). Using \(f_x = 4 x^3\), \(f_{xx} = 12 x^2\),
\(\sigma = 1\):
\(d(B_t^4) = 4 B_t^3\, dB_t + 6 B_t^2\, dt\), so
\(B_T^4 = 4\!\int_0^T B_s^3\, dB_s + 6\!\int_0^T B_s^2\, ds\) (since \(B_0 = 0\)).
Equivalently \(4\!\int_0^T B_s^3\, dB_s = B_T^4 - 6\!\int_0^T B_s^2\, ds\).

```python
T, N, M = 1.0, 2000, 5000
dt = T / N
rng = np.random.default_rng(4)
dW = rng.normal(0.0, np.sqrt(dt), size=(M, N))
B = np.concatenate([np.zeros((M, 1)), np.cumsum(dW, axis=1)], axis=1)

lhs = 4 * (B[:, :-1]**3 * dW).sum(axis=1)
int_B2 = ((B[:, :-1]**2) * dt).sum(axis=1)
rhs = B[:, -1]**4 - 6 * int_B2
print("mean |lhs - rhs|  :", np.mean(np.abs(lhs - rhs)))
print("max  |lhs - rhs|  :", np.max(np.abs(lhs - rhs)))
print("E[B_T^4]           :", (B[:, -1]**4).mean(), " (theory 3 T^2 =", 3*T**2, ")")
```

**5.** Euler--Maruyama vs exact GBM:

```python
mu, sigma, S0, T = 0.10, 0.40, 50.0, 1.0
N, M = 500, 10000
dt = T / N
rng = np.random.default_rng(5)
dW = rng.normal(0.0, np.sqrt(dt), size=(M, N))

S = np.zeros((M, N + 1)); S[:, 0] = S0
for k in range(N):
    S[:, k+1] = S[:, k] + mu*S[:, k]*dt + sigma*S[:, k]*dW[:, k]

B_T = dW.sum(axis=1)
S_ex = S0 * np.exp((mu - 0.5*sigma**2)*T + sigma*B_T)

for name, arr in [("Euler", S[:, -1]), ("Exact", S_ex)]:
    q = np.quantile(arr, [0.05, 0.95])
    print(f"{name:6s} mean={arr.mean():8.3f} var={arr.var(ddof=1):9.3f} "
          f"5%={q[0]:7.2f} 95%={q[1]:7.2f}")
print("theory E[S_T]     :", S0 * np.exp(mu*T))
```
Euler--Maruyama matches the *distribution shape* and quantiles well but slightly
underestimates \(E[S_T]\): the scheme effectively exponentiates a discretized
\(B_T \approx \sum dW_k\) (correct) but compounds \(S_k\) multiplicatively in
a way that injects an \(O(\sigma^2 \Delta t)\) discretization bias per step —
the very \(-\tfrac12 \sigma^2\) Ito correction that the exact log-formula
bakes in. The exact solution is therefore unbiased; Euler is \(O(\Delta t)\)
biased in the mean. Reducing `dt` shrinks the discrepancy.

</details>
